# ChEBI BioT5 Collection On Kaggle

This notebook clones the repo, ensures the ChEBI-20 dataset exists locally, runs the BioT5 grouped collection pipeline, and exports a clean artifact bundle for downstream notebooks.


In [ ]:
from pathlib import Path
import sys

REPO_URL = "https://github.com/mruniverse8/Thesis.git"
REPO_BRANCH = "gflownet"
REPO_DIR = Path("/kaggle/working/Thesis")
STAGE_NAME = "collect_chebi_biot5"

%cd /kaggle/working
!if [ -d "{REPO_DIR / '.git'}" ]; then echo "Reusing {REPO_DIR}"; elif [ -d "{REPO_DIR}" ]; then echo "Existing non-git directory at {REPO_DIR}; delete it and rerun the notebook." && false; else git clone --depth 1 "{REPO_URL}" "{REPO_DIR}"; fi
!git -C "{REPO_DIR}" fetch --depth 1 origin "{REPO_BRANCH}" && git -C "{REPO_DIR}" checkout -B "{REPO_BRANCH}" FETCH_HEAD

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from thesis_kaggle_support import (
    ensure_paths_exist,
    ensure_repo_selfies_vocab,
    ensure_runtime_dependencies,
    dump_yaml,
    export_stage_artifacts,
    json_dumps,
    load_yaml,
    read_json,
    report_runtime,
)

from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
import os

user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HF_TOKEN")
login(token=secret_value_0)

In [ ]:
TARGET_MOLECULES_PER_DESCRIPTION = 32
MAX_DESCRIPTIONS = 128
CHEBI_OUTPUT_DIR = REPO_DIR / "data" / "chebi20"
COLLECTION_OUTPUT_DIR = REPO_DIR / "data_collection" / "outputs" / "kaggle_chebi20_biot5_train"
DERIVED_TRAIN_FILE = REPO_DIR / "data" / "post_training" / "processed" / "train_multimol.jsonl"
TEMP_CONFIG_PATH = REPO_DIR / "kaggle" / "generated_configs" / "collect_biot5_chebi20.kaggle.yaml"

ensure_runtime_dependencies(REPO_DIR)
runtime_report = report_runtime(require_gpu=True)
vocab_path = ensure_repo_selfies_vocab(REPO_DIR)
print(json_dumps({
    "runtime": runtime_report,
    "vocab_path": str(vocab_path),
    "collection_output_dir": str(COLLECTION_OUTPUT_DIR),
    "derived_train_file": str(DERIVED_TRAIN_FILE),
}))


In [ ]:
CHEBI_PROCESSED_DIR = CHEBI_OUTPUT_DIR / "processed"
have_processed = all((CHEBI_PROCESSED_DIR / f"{split}.jsonl").exists() for split in ("train", "validation", "test"))
print(json_dumps({
    "have_processed": have_processed,
    "processed_dir": str(CHEBI_PROCESSED_DIR),
}))

%cd {REPO_DIR}
!if [ -f "{CHEBI_PROCESSED_DIR / 'train.jsonl'}" ] && [ -f "{CHEBI_PROCESSED_DIR / 'validation.jsonl'}" ] && [ -f "{CHEBI_PROCESSED_DIR / 'test.jsonl'}" ]; then echo "ChEBI processed splits already exist"; else python scripts/download_chebi20.py --output-dir "{CHEBI_OUTPUT_DIR}"; fi

config = load_yaml(REPO_DIR / "configs" / "collect_biot5_chebi20.yaml")
config["model"]["selfies_vocab_path"] = str(vocab_path)
config["data"]["train_file"] = str(CHEBI_PROCESSED_DIR / "train.jsonl")
config["data"]["staging_dir"] = str(COLLECTION_OUTPUT_DIR)
config["data"]["derived_train_file"] = str(DERIVED_TRAIN_FILE)
config["generation"]["target_molecules_per_description"] = int(TARGET_MOLECULES_PER_DESCRIPTION)
config.setdefault("runtime", {})["description_offset"] = 0
config["runtime"]["max_descriptions"] = None if MAX_DESCRIPTIONS is None else int(MAX_DESCRIPTIONS)
dump_yaml(config, TEMP_CONFIG_PATH)
print(f"Wrote config: {TEMP_CONFIG_PATH}")
print(TEMP_CONFIG_PATH.read_text(encoding="utf-8"))


In [ ]:
%cd {REPO_DIR}
!python scripts/collect_biot5_chebi20.py --config "{TEMP_CONFIG_PATH}"


In [ ]:
required_outputs = ensure_paths_exist({
    "chebi_processed_dir": CHEBI_PROCESSED_DIR,
    "collection_output_dir": COLLECTION_OUTPUT_DIR,
    "collection_summary": COLLECTION_OUTPUT_DIR / "summary.json",
    "derived_train_file": DERIVED_TRAIN_FILE,
})
artifact_dir, manifest = export_stage_artifacts(
    stage_name=STAGE_NAME,
    artifact_map={
        "chebi20_processed": CHEBI_PROCESSED_DIR,
        "collection_outputs": COLLECTION_OUTPUT_DIR,
        "post_training_processed/train_multimol.jsonl": DERIVED_TRAIN_FILE,
    },
    metadata={
        "required_outputs": required_outputs,
        "config_path": str(TEMP_CONFIG_PATH),
        "summary": read_json(COLLECTION_OUTPUT_DIR / "summary.json"),
    },
)
print(json_dumps({
    "artifact_dir": str(artifact_dir),
    "manifest": manifest,
}))
